In [12]:
import pandas as pd 
import numpy as np
import re

from astroquery.simbad import Simbad

In [ ]:
EXPLICIT_ALIASES = {
    'AUMic': ['AU Mic'],
    'EpsilonIndi': ['eps Ind', 'Epsilon Indi'],
    'piMensae': ['pi Men', 'Pi Mensae'],
}


def candidate_names(name):
    name = str(name).strip()
    candidates = [name]
    candidates.extend(EXPLICIT_ALIASES.get(name, []))

    patterns = [
        (r'^HD(\\d+)$', r'HD \\1'),
        (r'^HIP(\\d+)$', r'HIP \\1'),
        (r'^GJ(\\d+)$', r'GJ \\1'),
        (r'^(55)CncA$', r'55 Cnc'),
    ]

    for pattern, replacement in patterns:
        if re.match(pattern, name, flags=re.IGNORECASE):
            candidates.append(re.sub(pattern, replacement, name, flags=re.IGNORECASE))

    seen = set()
    ordered_candidates = []
    for candidate in candidates:
        if candidate not in seen:
            seen.add(candidate)
            ordered_candidates.append(candidate)

    return ordered_candidates


def fetch_gaia_dr3_id(name):
    for candidate in candidate_names(name):
        ids = Simbad.query_objectids(candidate)
        if ids is None:
            continue

        for row in ids:
            object_id = str(row[0])
            if object_id.startswith('Gaia DR3 '):
                return pd.Series({
                    'matched_name': candidate,
                    'gaia_dr3_id': object_id.replace('Gaia DR3 ', '')
                })

    return pd.Series({
        'matched_name': pd.NA,
        'gaia_dr3_id': pd.NA
    })


#stars_df[['matched_name', 'gaia_dr3_id']] = stars_df['star'].apply(fetch_gaia_dr3_id)

In [14]:
sample_df = pd.read_csv('sample.csv')
sample_df.head(5)

,star,matched_name,gaia_dr3_id
0,55CncA,55CncA,704967037090946688
1,AUMic,AU Mic,6794047652729201024
2,EpsilonIndi,eps Ind,6412595290592307840
3,GJ436,GJ436,4017860992519744384
4,HAT-P-26,HAT-P-26,3668036348641580288
